# jupyter-loopback end-to-end demo

Everything is inline in this notebook. Read each cell to see exactly what integrating `jupyter-loopback` takes. Three moving parts:

1. **A jupyter-server extension** (the `loopback_demo._jupyter` package on disk) mounts the proxy.
2. **An in-kernel HTTP+WS server** (defined right below) serves some content.
3. **`autodetect_prefix`** tells the kernel which browser-reachable URL to hand out.

If the red square renders and the WebSocket echo answers back, the full round-trip works.

## 0. Enable the comm bridge (for VS Code and other non-jupyter-server frontends)

JupyterLab / Hub / Binder can reach the loopback port through the HTTP proxy mounted by the server extension (step 2), so this step is not strictly required there. But for VS Code Jupyter (local or SSH), Colab, Shiny, Solara, and marimo, the notebook webview has no way to reach the jupyter-server origin for root-relative URLs. Enabling the comm bridge gives the browser a second, always-available path to the kernel's loopback port over the kernel's comm channel.

Once enabled, `window.__jupyter_loopback__` is installed in the output pane and exposes `fetch(port, path)`, `resolveUrl(port, path, {mime})`, and `openWebSocket(port, path)`. The HTML cell below feature-detects those APIs and prefers them; otherwise it falls back to the direct URL that the HTTP proxy handles.

In [ ]:
import jupyter_loopback

jupyter_loopback.enable_comm_bridge()

## 1. What the kernel sees

`jupyter-loopback` reads four environment variables. `JPY_SESSION_NAME` (or `JPY_PARENT_PID`) proves we're in a kernel; `JUPYTERHUB_SERVICE_PREFIX` or `JPY_BASE_URL` carry any per-user URL prefix.

In [ ]:
import os

for var in ("JPY_SESSION_NAME", "JPY_PARENT_PID", "JUPYTERHUB_SERVICE_PREFIX", "JPY_BASE_URL"):
    print(f"{var} = {os.environ.get(var)!r}")

## 2. Extension check

The `loopback_demo` package installs a one-line jupyter-server extension that calls `setup_proxy_handler(web_app, namespace='loopback-demo')`. When the Docker image boots JupyterLab, the extension auto-loads because the package ships `jupyter-config/jupyter_server_config.d/loopback-demo.json`.

In [ ]:
import inspect

from loopback_demo import _jupyter

print(inspect.getsource(_jupyter))

## 3. In-kernel HTTP + WebSocket server

A small Tornado app. Three routes:

- `GET /hello` returns JSON.
- `GET /image.png` returns a 1×1 red PNG. The HTML display below upscales it so it's visible.
- `WS  /ws` echoes messages back.

Handlers set `Access-Control-Allow-Origin: *` so the browser can embed responses in iframes (e.g. if you ever render this through a folium HTML output).

In [ ]:
import asyncio
import json
import threading
import time

from tornado.httpserver import HTTPServer
from tornado.ioloop import IOLoop
from tornado.testing import bind_unused_port
from tornado.web import Application, RequestHandler
from tornado.websocket import WebSocketHandler

# Minimal 1x1 red PNG, hand-rolled so the demo has no Pillow dep.
RED_PNG = (
    b"\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\x01\x00\x00\x00"
    b"\x01\x08\x02\x00\x00\x00\x90wS\xde\x00\x00\x00\x0cIDATx\x9cc\xf8"
    b"\xcf\xc0\x00\x00\x00\x03\x00\x01\x84\xd0\xf9]\x00\x00\x00\x00IEND"
    b"\xaeB`\x82"
)


class Hello(RequestHandler):
    def get(self):
        self.set_header("Access-Control-Allow-Origin", "*")
        self.set_header("Content-Type", "application/json")
        self.write(json.dumps({"message": "hello from the kernel", "now": time.time()}))


class Image(RequestHandler):
    def get(self):
        self.set_header("Access-Control-Allow-Origin", "*")
        self.set_header("Content-Type", "image/png")
        self.write(RED_PNG)


class EchoWS(WebSocketHandler):
    def check_origin(self, origin):
        return True

    def open(self):
        self.count = 0

    async def on_message(self, message):
        self.count += 1
        await self.write_message(f"[{self.count}] {message}")


# Bind a loopback port up front so we know the number before starting.
sock, port = bind_unused_port(address="127.0.0.1")
print(f"server will bind to 127.0.0.1:{port}")


def run():
    asyncio.set_event_loop(asyncio.new_event_loop())
    loop = IOLoop.current()
    app = Application(
        [
            (r"/hello", Hello),
            (r"/image.png", Image),
            (r"/ws", EchoWS),
        ]
    )
    server = HTTPServer(app)
    server.add_sockets([sock])
    ready.set()
    loops.append(loop)
    loop.start()


ready = threading.Event()
loops = []  # one-element list so run() can publish the loop back to us
thread = threading.Thread(target=run, name="demo-server", daemon=True)
thread.start()
ready.wait(timeout=5.0)
print("server running")

## 4. Build a browser-reachable URL

`autodetect_prefix('loopback-demo')` returns a root-absolute path template. `None` outside a Jupyter kernel so the same code can run locally against `127.0.0.1:<port>` directly.

In [ ]:
from jupyter_loopback import autodetect_prefix

prefix = autodetect_prefix("loopback-demo")
print("prefix template:", prefix)

http_url = f"{prefix.format(port=port)}" if prefix else f"http://127.0.0.1:{port}"
ws_url = f"{http_url}/ws"
print("http_url:", http_url)
print("ws_url:  ", ws_url)

## 5. Sanity check: kernel hits the loopback socket

The kernel and the server share a process, so this goes straight through `127.0.0.1`. If this fails, the server is broken and the proxy tests below will too.

In [ ]:
import requests

r = requests.get(f"http://127.0.0.1:{port}/hello", timeout=5)
print("status:", r.status_code)
print("body:  ", r.json())

## 6. Browser round-trip

The HTML below feature-detects `window.__jupyter_loopback__`:

- If the comm bridge is enabled (step 0), the browser fetches `/hello`, `/image.png`, and opens the WebSocket **through the kernel comm channel** via `api.fetch`, `api.resolveUrl`, and `api.openWebSocket`. This is the path that works in VS Code Jupyter (including Remote-SSH), Colab, Shiny, Solara, and marimo.
- Otherwise it uses the direct URL built in step 4, which goes through the jupyter-server's HTTP proxy handler. This is the faster path when it's available (JupyterLab, Hub, Binder, Notebook 7+).

Three checks regardless of path:

- **JSON fetch**: text of `GET /hello` appears inline.
- **Inline image**: 1×1 red PNG, CSS-upscaled to 80×80. If you see a red square, binary bodies survive the round-trip.
- **WebSocket echo**: type a message, press Enter. Server echoes back prefixed with a count.

In [ ]:
from IPython.display import HTML

# Pick a path: if autodetect produced a prefix (JupyterLab / Hub) we can
# fetch directly through the Path A proxy. Otherwise we'll still compute
# a direct loopback URL as a fallback, but the HTML below prefers the
# comm bridge (window.__jupyter_loopback__) when it's available.
http_fallback = http_url
ws_fallback = ws_url

html = f"""
<div style="border:1px solid #ddd;border-radius:6px;padding:12px;
            font-family:system-ui,sans-serif;max-width:640px">
  <p style="margin:0 0 8px">Bound to <code>127.0.0.1:{port}</code> in the kernel.</p>
  <ul style="margin:0 0 8px">
    <li><code>GET /hello</code> <span id="lpbk-hello-{port}">(loading...)</span></li>
    <li><code>GET /image.png</code>
      <img id="lpbk-img-{port}" alt="red square"
           style="vertical-align:middle;width:80px;height:80px;
                  image-rendering:pixelated;border:1px solid #ccc;
                  border-radius:4px;margin-left:6px"/></li>
    <li><code>WS /ws</code> (<span id="lpbk-mode-{port}">detecting...</span>)</li>
  </ul>
  <input id="lpbk-in-{port}" type="text" placeholder="type, press Enter"
         style="width:100%;padding:6px;border:1px solid #ccc;border-radius:4px"/>
  <pre id="lpbk-log-{port}"
       style="margin-top:8px;background:#f7f7f7;padding:8px;border-radius:4px;
              max-height:140px;overflow:auto"></pre>
  <script>
    (async function() {{
      var port = {port};
      var httpFallback = {json.dumps(http_fallback)};
      var wsFallback = {json.dumps(ws_fallback)};
      var api = window.__jupyter_loopback__;
      var useBridge = !!(api && api.resolveUrl && api.openWebSocket);

      var log = document.getElementById("lpbk-log-" + port);
      var img = document.getElementById("lpbk-img-" + port);
      var helloSpan = document.getElementById("lpbk-hello-" + port);
      var modeSpan = document.getElementById("lpbk-mode-" + port);
      modeSpan.textContent = useBridge ? "comm bridge" : "direct";

      // GET /hello
      try {{
        if (useBridge) {{
          var resp = await api.fetch(port, "/hello");
          var text = await resp.text();
          helloSpan.textContent = text;
        }} else {{
          var resp2 = await fetch(httpFallback + "/hello");
          helloSpan.textContent = await resp2.text();
        }}
      }} catch (err) {{
        helloSpan.textContent = "error: " + err;
      }}

      // GET /image.png
      try {{
        if (useBridge) {{
          img.src = await api.resolveUrl(port, "/image.png", {{mime: "image/png"}});
        }} else {{
          img.src = httpFallback + "/image.png";
        }}
      }} catch (err) {{
        log.textContent += "[image error: " + err + "]\\n";
      }}

      // WS /ws
      var ws;
      if (useBridge) {{
        ws = api.openWebSocket(port, "/ws");
      }} else {{
        var abs = new URL(wsFallback, window.location.origin);
        abs.protocol = window.location.protocol === "https:" ? "wss:" : "ws:";
        ws = new WebSocket(abs.toString());
      }}
      ws.onopen    = function()    {{ log.textContent += "[open]\\n"; }};
      ws.onclose   = function()    {{ log.textContent += "[close]\\n"; }};
      ws.onerror   = function(e)   {{ log.textContent += "[error]\\n"; }};
      ws.onmessage = function(ev)  {{
        log.textContent += "< " + ev.data + "\\n";
        log.scrollTop = log.scrollHeight;
      }};

      var input = document.getElementById("lpbk-in-" + port);
      input.addEventListener("keydown", function(e) {{
        if (e.key !== "Enter") return;
        var v = input.value; input.value = "";
        log.textContent += "> " + v + "\\n";
        log.scrollTop = log.scrollHeight;
        ws.send(v);
      }});
    }})();
  </script>
</div>
"""

HTML(html)

## 7. Cleanup

Stop the background IOLoop. In a real library you'd do this in a shutdown hook.

In [ ]:
loops[0].add_callback(loops[0].stop)
thread.join(timeout=5)
print("stopped")